In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../../day02/lab06_clean-dataset/results/secom_clean.csv')

sensor_cols = [c for c in df.columns if c.startswith('sensor_')]
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

df['불량여부'] = (df['result'] == '불량').astype(int)

X = df[sensor_cols]
y = df['불량여부']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('학습용:', X_train.shape[0], '행, 불량', int(y_train.sum()), '건')
print('시험용:', X_test.shape[0], '행, 불량', int(y_test.sum()), '건')

학습용: 1253 행, 불량 83 건
시험용: 314 행, 불량 21 건


[Step 0 결과]<br>
secom_clean.csv 를 df로 불러오고, 센서 열 빈칸은 중앙값으로 채웠다<br>
result → 불량여부(불량=1, 양품=0) 열 추가<br>
X = 센서 열, y = 불량여부<br>
test_size=0.2, random_state=42, stratify=y 로 분할<br>
학습용 : [1253]행, 불량 [83]건<br>
시험용 : [314]행, 불량 [21]건<br>
→ lab07과 [같은] 숫자로, 같은 데이터·같은 기준으로 나눴음을 확인했다

## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 비교할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 기준 모델 | 학습을 전혀 하지 않고 늘 같은 답만 내놓는 모델. 비교의 바닥선이 된다 |
| 학습 | 답이 붙은 기록을 넣어 규칙을 찾게 하는 일 |
| 예측 | 처음 보는 기록에 답을 붙이는 일 |
| 정확도 | 전체 중 맞힌 비율. 오늘 쓰는 유일한 점수이고, 내일 이 점수를 의심하게 된다 |

## Step 2. 게으름뱅이 모델 만들기

In [2]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(양품)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.zeros(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델이 불량이라 한 건수:", 기준예측.sum())
print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")

기준 모델이 불량이라 한 건수: 0
기준 모델 정확도: 93.31 %


In [3]:
# 시험용에서 양품이 몇 건, 불량이 몇 건인지
print("시험용 양품:", (y_test == 0).sum(), "건")
print("시험용 불량:", (y_test == 1).sum(), "건")

# 전부 양품이라 답하면 -> 양품은 다 맞고, 불량은 다 틀린다
print("맞힌 것:", (y_test == 0).sum(), "/", len(y_test))

시험용 양품: 293 건
시험용 불량: 21 건
맞힌 것: 293 / 314


[기준 모델이 높은 점수를 받는 이유]<br>
시험용 [314]건 중 양품이 [293]건이다.<br>
전부 양품이라 답하면 [293]건은 자동으로 맞는다.<br>
불량 [21]건은 전부 놓치지만, 개수가 적어 점수에 거의 영향이 없다.

## Step 4. 모델을 추천받기

[추천받은 모델]<br>
1. [로지스틱 회귀] - [둘 중 하나를 고르는 문제의 기본이고, 어느 열이 얼마나 작용했는지 볼 수 있다]<br>
2. [의사결정나무] - [자르는 기준이 눈에 보여서 설명하기 쉽다]<br>
내가 고른 것 : [로지스틱 회귀]

In [4]:
# 오늘은 손보지 않은 기본 설정 그대로 — class_weight 등 불균형 보정 없음
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

모델들 = {
    '로지스틱 회귀': LogisticRegression(),
    '의사결정나무': DecisionTreeClassifier(random_state=42),
}

결과 = []
for 이름, model in 모델들.items():
    model.fit(X_train, y_train)
    예측 = model.predict(X_test)
    정확도 = (예측 == y_test).mean()
    결과.append({
        '모델': 이름,
        '불량이라고 예측한 건수': int((예측 == 1).sum()),
        '정확도(%)': round(정확도 * 100, 2),
    })

비교표 = pd.DataFrame(결과).set_index('모델')
print('기준 모델(전부 양품) 정확도: 93.31 %')
비교표

기준 모델(전부 양품) 정확도: 93.31 %


C:\Users\13sx0\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,불량이라고 예측한 건수,정확도(%)
모델,,
로지스틱 회귀,0,93.31
의사결정나무,31,86.62


Step 5: 추천 모델 학습시키고 점수 재기

In [5]:
# 로지스틱 회귀는 스케일에 민감한 모델이라 표준화를 함께 넣는다
# (기준은 X_train으로만 잡고, X_test는 그 기준으로 변환만 한다)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_표준화 = scaler.fit_transform(X_train)
X_test_표준화 = scaler.transform(X_test)

# 불균형 보정 없이, 기본 설정 그대로
model = LogisticRegression()
model.fit(X_train_표준화, y_train)

예측 = model.predict(X_test_표준화)

정확도 = (예측 == y_test).mean()
불량_예측_건수 = int((예측 == 1).sum())
그중_실제_불량 = int(((예측 == 1) & (y_test == 1)).sum())

print('1. 정확도:', round(정확도 * 100, 2), '%')
print('2. 불량이라고 예측한 건수:', 불량_예측_건수)
print('3. 그중 실제로 불량이었던 건수:', 그중_실제_불량)

1. 정확도: 93.95 %
2. 불량이라고 예측한 건수: 2
3. 그중 실제로 불량이었던 건수: 2


## Step 6. 모델 기록표

| 모델 | 왜 썼나 | 정확도 | 불량이라 한 건수 | 그중 진짜 |
|---|---|---|---|---|
| 기준 모델 (전부 양품) | 비교할 바닥선 | [93.31]% | [0] | [0] |
| [로지스틱 회귀] | [분류의 기본이고 결과를 설명하기 쉬워서] | [93.95]% | [2] | [2] |

---
## 직접 해보기 (도전) - 게으름뱅이를 반대로 만들면

- 상황: 전부 양품이라 답하는 모델을 만들어봤다. 반대는 어떨까
- 할 일: 전부 불량이라 답하는 모델의 점수를 재고, 추천 모델을 하나 더 붙여 표를 늘린다
- 결과물: 네 줄짜리 기록표 1개

In [6]:
# 아직 안 써본 추천 모델 — 의사결정나무 (트리는 스케일에 안 민감해서 표준화 없이 원래 X_train 그대로 사용)
예측_트리 = DecisionTreeClassifier(random_state=42).fit(X_train, y_train).predict(X_test)

# 1. 전부 양품이라 답하는 기준 모델
예측_전부양품 = np.zeros(len(y_test), dtype=int)

# 2. 전부 불량이라 답하는 모델
예측_전부불량 = np.ones(len(y_test), dtype=int)


def 요약(이름, 예측값):
    return {
        '모델': 이름,
        '정확도(%)': round((예측값 == y_test).mean() * 100, 2),
        '불량이라 한 건수': int((예측값 == 1).sum()),
        '그중 진짜 불량 건수': int(((예측값 == 1) & (y_test == 1)).sum()),
    }


기록표 = pd.DataFrame([
    요약('전부 양품 모델 (기준)', 예측_전부양품),
    요약('전부 불량 모델', 예측_전부불량),
    요약('로지스틱 회귀 (앞에서 학습)', 예측),
    요약('의사결정나무 (방금 학습)', 예측_트리),
]).set_index('모델')

기록표

,정확도(%),불량이라 한 건수,그중 진짜 불량 건수
모델,,,
전부 양품 모델 (기준),93.31,0,0
전부 불량 모델,6.69,314,21
로지스틱 회귀 (앞에서 학습),93.95,2,2
의사결정나무 (방금 학습),86.62,31,5
